# Run a test client in Openshift

## Apply the feature store definitions

We'll mount the `feature_store.yaml` ConfigMap created by the Operator within a Kubernetes `Deployment` to run `feast apply`.

Then we create the client `Deployment` to apply the definitions, according to the [client-openshift.yaml](./client-openshift.yaml) manifest

We are going to emulate the `feast init -t postgres sample` command using Python code in an initContainer at client `Deployment` runtime. This is needed to mock the request of additional
parameters to configure the DB connection and also request the upload of example data to Postgres tables.

In [ ]:
!kubectl apply -f client-openshift.yaml

Monitoring the log of the `Deployment`.

In [ ]:
!kubectl wait --for=condition=available deployment/client --timeout=2m
!kubectl logs deploy/client -c feast-apply

Verify the client `feature_store.yaml`.

In [ ]:
!kubectl exec deploy/client -c client -- cat feature_store.yaml

## Run test code

Finally, we run the full test suite from the client folder.

In [ ]:
!kubectl exec deploy/client -c client -- python3 test_workflow.py

Note If you see the following error, it is likely due to the [issue #4392](https://github.com/feast-dev/feast/issues/4392):
Remote registry client does not map application errors:

```
Feature view driver_hourly_stats_fresh does not exist in project sample
```